# Trajectory Cache Backend Benchmark

Comparison of four trajectory-cache backends as approximate-nearest-neighbor
(ANN) search structures over MoveIt `PlanRequest`s.

Each request is reduced to a feature vector (joint angles, optionally pose),
and the cache returns any previously-planned trajectory whose stored request
lies within configured per-coordinate tolerances of the query.

Workflow per backend:

1. **Collect phase** — sample random goals around `idle`, plan & execute
   `idle -> goal -> idle` until 25 round-trips succeed.
2. **Cycle phase** — replay the same successful goals 5 times. A working
   cache should hit on every leg after the first cycle.

Per leg we log: plan/query time, execution time, and whether the trajectory
came from the cache (`from_cache`) or was planned fresh.

In [ ]:
import os

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

In [ ]:
CSV_PATH = os.path.expandvars("$TABLETOP_CACHE_DIR/cache_benchmark.csv")

df = pd.read_csv(CSV_PATH)
df = df[df["success"]].reset_index(drop=True)
df["plan_time_ms"] = df["plan_time_s"] * 1000

BACKEND_ORDER = ["lmdb", "dict", "linear", "kdtree"]
BACKEND_COLORS = {
    "lmdb": "#1f77b4",
    "dict": "#aec7e8",
    "linear": "#2ca02c",
    "kdtree": "#ff7f0e",
}

print(f"Loaded {len(df)} successful legs from {CSV_PATH}")
display(df.describe())
display(df)

## Backend overview

All four backends implement the same `Mapping`-like API
(`__setitem__` / `__getitem__` / `__contains__` / `__delitem__`) over
`PlanRequest -> RobotTrajectory`. They differ only in how they index
the request space.

Notation: `N` = entries in the cache, `B` = candidate trajectories per
match group (bounded by `max_trajectories`), `d` = feature-vector
dimensionality (12 or 13).

### LMDB fuzzy cache (`LMDBFuzzyTrajectoryCache`)

- **Concept.** Quantize every float coordinate of the request fingerprint
  via `int(x // tolerance)` to produce a deterministic integer-bin key.
  JSON-encode the bin dict and use the resulting bytes as an LMDB key.
  Each bin holds up to `B` candidates sorted by `path_cost`.
- **Update.** `O(log B)` to bisect-insert into the per-bin sorted list,
  plus one LMDB write transaction (disk I/O).
- **Query.** `O(1)` hash lookup on the bytes key + `O(B)` to read the
  ranked candidates.
- **Space.** `O(U)` on disk, where `U` is the number of distinct fuzzy
  bins ever populated.
- **Caveat.** *Bin-boundary aliasing* — coordinates that fall near
  `k * tolerance` toggle into adjacent bins under tiny floating-point
  drift, so the same physical request can produce different keys across
  runs.

### Dict fuzzy cache (`DictFuzzyTrajectoryCache`)

- **Concept.** Identical indexing strategy to LMDB, but the bin map is a
  Python `dict` instead of an LMDB env. The dict is pickled on `close()`
  and unpickled on `open()`.
- **Update.** `O(log B)` bisect-insert (no per-operation disk I/O during
  the session — only on `close()`).
- **Query.** `O(1)` Python `dict` lookup + `O(B)` to read candidates.
- **Space.** `O(U)` in memory + `O(U)` on disk after `close()`.
- **Caveat.** Same bin-boundary aliasing as LMDB.

### Linear brute-force cache (`LinearTrajectoryCache`)

- **Concept.** Store each request fingerprint verbatim (no quantization).
  On query, scan **every** stored fingerprint and apply per-coordinate
  tolerance checks `|x_i - y_i| <= tolerance_i`.
- **Update.** `O(1)` amortized — dict insertion under the exact JSON-bytes
  fingerprint (byte-identical duplicates merge, near-but-not-identical
  ones don't).
- **Query.** `O(N \cdot d)` — full scan, comparing `d` coordinates per
  entry.
- **Space.** `O(N)`.
- **Strength.** Recall is exactly the configured tolerance — no aliasing.
- **Weakness.** Query cost grows linearly with cache size; the brute-force
  baseline an ANN structure should beat.

### K-d tree cache (`KDTreeTrajectoryCache`)

- **Concept.** Map each request to a 12-D feature vector (joint-space
  goals: 6 start joints ++ 6 goal joints) or 13-D (Cartesian goals: 6
  start joints ++ 3 position ++ 4 quaternion). Pre-scale each coordinate
  by `1 / tolerance` so the per-coordinate tolerance becomes a unit-radius
  L∞ ball. Build two scipy `KDTree`s (one per goal-type dimensionality)
  and query with `query_ball_point(scaled_query, r=1, p=np.inf)`.
- **Update.** `O(1)` amortized — append to the feature list; tree rebuild
  is deferred until the next query.
- **Query.** `O(\log N + k)` for a balanced ball query (`k` = matches),
  plus one `O(N \log N)` rebuild whenever the tree is stale.
- **Space.** `O(N \cdot d)` for features + `O(N)` for the tree.
- **Strength.** Preserves exact per-coordinate tolerance semantics (no
  aliasing) while achieving sublinear average-case query time.

## Plot 1 — Cache hit rate by backend and phase

The headline benchmark number: of all the legs the cache **could have**
served from memory, what fraction did it actually find?

- **Collect phase**: first-time visits. The reverse-trajectory caching
  (`cache_trajectory(_reverse=True)`) means the return leg of each
  round-trip should always be a cache hit — so the collect phase has a
  theoretical ceiling around 50%.
- **Cycle phase**: replays. Every leg should be a cache hit — the cache
  has seen this exact (start, goal) pair before.

In [ ]:
hit_rate = (
    df.groupby(["backend", "phase"])["from_cache"].mean() * 100
).unstack(level="phase")
hit_rate = hit_rate.reindex(BACKEND_ORDER)[["collect", "cycle"]]

fig, ax = plt.subplots(figsize=(8.5, 5))
x = np.arange(len(BACKEND_ORDER))
w = 0.38
ax.bar(
    x - w / 2, hit_rate["collect"], width=w, label="collect", color="#aec7e8"
)
ax.bar(x + w / 2, hit_rate["cycle"], width=w, label="cycle", color="#1f77b4")
for i, b in enumerate(BACKEND_ORDER):
    ax.text(
        i - w / 2,
        hit_rate.loc[b, "collect"] + 1.5,  # type: ignore
        f"{hit_rate.loc[b, 'collect']:.0f}%",
        ha="center",
        fontsize=9,
    )
    ax.text(
        i + w / 2,
        hit_rate.loc[b, "cycle"] + 1.5,  # type: ignore
        f"{hit_rate.loc[b, 'cycle']:.0f}%",
        ha="center",
        fontsize=9,
    )
ax.set_xticks(x)
ax.set_xticklabels(BACKEND_ORDER)
ax.set_ylabel("Cache hit rate (%)")
ax.set_xlabel("Backend")
ax.set_ylim(0, 110)
ax.axhline(
    50,
    color="gray",
    linestyle=":",
    alpha=0.5,
    label="collect ceiling (reverse-leg only)",
)
ax.axhline(100, color="gray", linestyle="-", alpha=0.3)
ax.legend(loc="lower right")
ax.set_title("Cache hit rate by backend × phase")
plt.tight_layout()
plt.show()

**Takeaway.** Linear and k-d tree achieve **100% hit rate** in the cycle
phase — every replay is served from cache. The two fuzzy backends miss
~25-35% of cycle-phase hits because their bin-boundary aliasing fragments
the same physical request across multiple keys when joint values land
near round numbers (idle has joints at exactly `0` and `±π/2`).

## Plot 2 — Plan / query time distribution: hit vs miss

A cache hit's value lies in how much cheaper it is than a fresh plan.
Plotted on a log scale to make the gap visible — the two distributions
are typically two orders of magnitude apart.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
positions: list[float] = []
tick_pos: list[float] = []
tick_labels: list[str] = []
data: list[np.ndarray] = []
colors: list[str] = []
for i, backend in enumerate(BACKEND_ORDER):
    for j, hit in enumerate([False, True]):
        d = df.loc[
            (df.backend == backend) & (df.from_cache == hit), "plan_time_ms"
        ].values
        if len(d) == 0:
            continue
        pos = i * 3 + j
        positions.append(pos)
        data.append(d)  # type: ignore
        colors.append("#d62728" if not hit else "#2ca02c")
    tick_pos.append(i * 3 + 0.5)
    tick_labels.append(backend)

bp = ax.boxplot(
    data,
    positions=positions,
    widths=0.7,
    patch_artist=True,
    showfliers=True,
    flierprops={"marker": ".", "alpha": 0.5},
)
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.55)
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels)
ax.set_yscale("log")
ax.set_ylabel("Plan / query time (ms, log scale)")
ax.set_xlabel("Backend")
ax.set_title(
    "Plan/query time distribution: cache miss (red) vs cache hit (green)"
)
ax.grid(axis="y", which="both", alpha=0.25)

ax.legend(
    handles=[
        mpatches.Patch(
            color="#d62728", alpha=0.55, label="cache miss (fresh plan)"
        ),
        mpatches.Patch(color="#2ca02c", alpha=0.55, label="cache hit"),
    ],
    loc="upper right",
)
plt.tight_layout()
plt.show()

**Takeaway.** Cache hits are 50-150× faster than fresh plans across every
backend — the *value of a hit* is essentially the same regardless of
indexing strategy. What differentiates the backends is **how often they
deliver one** (Plot 1). The fresh-plan distribution is determined by
the OMPL planner's `planning_time` budget; cache-hit time is dominated
by the lookup overhead of each data structure.

## Plot 3 — Cumulative plan / query time

Wall-clock view: as the experiment progresses, how much time has the
robot spent in planning + cache lookups for each backend?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for backend in BACKEND_ORDER:
    d = df[df.backend == backend].reset_index(drop=True)
    cum = d.plan_time_s.cumsum()
    ax.plot(
        d.index + 1, cum, label=backend, color=BACKEND_COLORS[backend], lw=2
    )

# Mark the collect/cycle phase boundary per backend (same for all if
# `n_unique_goals * 2` collect legs).
lengths = df.groupby("backend")["phase"].value_counts().unstack().fillna(0)
if "collect" in lengths.columns:
    collect_n = int(lengths["collect"].iloc[0])
    ax.axvline(
        collect_n,
        color="gray",
        linestyle="--",
        alpha=0.4,
        label=f"collect → cycle (leg {collect_n})",
    )
ax.set_xlabel("Leg index (collect first, then cycles)")
ax.set_ylabel("Cumulative plan / query time (s)")
ax.set_title("Total time spent in plan/query across the benchmark")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Takeaway.** During collect, all four backends accumulate planning time
at roughly the same rate — most legs miss and pay the full OMPL budget.
After the collect/cycle boundary the slopes diverge sharply: linear and
k-d tree flatten out (cache serves every leg in microseconds), while
lmdb and dict continue to accumulate planning cost on the ~30% of cycle
legs they fail to recognize. Over a long-running experiment this gap
compounds into many minutes of wasted compute.

## Plot 4 — Hit rate by replay cycle

Does cache hit rate improve as the experiment repeats the same goal
sequence, or is the gap fixed by the indexing strategy?

In [ ]:
cyc = df[df.phase == "cycle"].copy()
hr = (cyc.groupby(["backend", "cycle"])["from_cache"].mean() * 100).unstack(
    level="backend"
)
hr = hr[BACKEND_ORDER]

fig, ax = plt.subplots(figsize=(8.5, 5))
for backend in BACKEND_ORDER:
    ax.plot(
        hr.index + 1,
        hr[backend],
        marker="o",
        lw=2,
        color=BACKEND_COLORS[backend],
        label=backend,
    )
ax.set_xlabel("Replay cycle")
ax.set_ylabel("Cache hit rate (%)")
ax.set_xticks(hr.index + 1)
ax.set_ylim(0, 105)
ax.set_title("Cache hit rate across replay cycles")
ax.axhline(100, color="gray", linestyle="-", alpha=0.3)
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Takeaway.** Linear and k-d tree saturate at 100% from cycle 1 onward
— once a goal has been planned, every subsequent replay hits. The fuzzy
backends never catch up: their hit rate is bounded by how often joint
values happen to *not* sit on bin boundaries. Adding more replays
doesn't help, because the misses aren't due to the cache being cold —
they're due to the indexing strategy fragmenting the same point across
multiple keys.

## Summary table

In [ ]:
summary = (
    df.groupby(["backend", "phase"])
    .agg(
        legs=("plan_time_s", "size"),
        hit_rate_pct=("from_cache", lambda s: 100 * s.mean()),
        avg_plan_ms=("plan_time_ms", "mean"),
        median_plan_ms=("plan_time_ms", "median"),
    )
    .reset_index()
)
summary["backend"] = pd.Categorical(
    summary["backend"], categories=BACKEND_ORDER, ordered=True
)
summary = summary.sort_values(["backend", "phase"]).reset_index(drop=True)
summary.style.format(
    {
        "hit_rate_pct": "{:.1f}%",
        "avg_plan_ms": "{:.2f}",
        "median_plan_ms": "{:.2f}",
    }
)

## Conclusions

1. **Hit rate is set by the indexing strategy, not by the storage backend.**
   LMDB-fuzzy and dict-fuzzy track each other to within 1-2 percentage
   points across every phase — they share the same `int(x // tolerance)`
   quantization. Whether bins live on disk or in RAM is irrelevant to
   recall.
2. **Per-coordinate tolerance comparison beats quantization.** Both linear
   and k-d tree achieve perfect recall in the cycle phase. They share
   the same matching predicate (`|x_i - y_i| <= tolerance_i`), just
   different indexing structures over the same equivalence class.
3. **A cache hit is worth ~100× a cache miss.** The value of an extra
   hit is large and roughly constant; the *cost* of false misses
   (fuzzy's aliasing) compounds across long-running experiments.
4. **K-d tree is the production-ready choice.** It matches linear's
   recall while keeping average-case query time sublinear in cache
   size — important once the cache holds thousands of trajectories.